# Ridge Decoder R² Across Time by Motor Speed

This notebook loads `ridge_decoder_predictions_motor.parquet` for all motor sessions
across both projects and plots how the decoder R² varies across time within a trial,
grouped by motor speed.

The key question is whether R² is larger or smaller during the initial ramp-up period
(before running speed reaches the motor-set constant speed).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.interpolate import interp1d
from tqdm import tqdm

from cottage_analysis.summary_analysis import get_session_list
from cottage_analysis.summary_analysis import load_project_predictions
from cottage_analysis.analysis import treadmill

# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"
}

In [ ]:
# Cell 2: Load predictions for both projects

df_ccyp = load_project_predictions("ccyp_l5_3d_vision", session_to_exclude=SESSIONS_TO_EXCLUDE.keys())
df_colasa = load_project_predictions("colasa_3d-vision_revisions", session_to_exclude=SESSIONS_TO_EXCLUDE.keys())

df_ccyp["project"] = "ccyp_l5_3d_vision"
df_colasa["project"] = "colasa_3d-vision_revisions"

predictions_df = pd.concat([df_ccyp, df_colasa], ignore_index=True)

print(f"Total predictions rows: {len(predictions_df)}")
print(f"Sessions: {predictions_df['session'].nunique()}")
print(f"Columns: {predictions_df.columns.tolist()}")

In [ ]:
# Cell 3: Sync sessions to get MotorSpeed from trials_df
#
# For each session, call treadmill.sync_all_recordings() with the same parameters
# used by the ridge decoder pipeline (acceleration_time=None, cut_trial_end=None,
# trial_duration=None) so we get the full uncut trials.
# Extract per-trial MotorSpeed and frame_rate.

PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]

motor_speed_records = []  # (session, trial_no, motor_speed, frame_rate)

for project in PROJECTS:
    flexilims_session = flz.get_flexilims_session(project_id=project)
    session_list = get_session_list.get_motor_session_list(flexilims_session)

    for sess_name in tqdm(session_list, desc=project):
        if sess_name in SESSIONS_TO_EXCLUDE:
            continue
        if sess_name not in predictions_df["session"].values:
            continue

        try:
            # Get frame rate from suite2p dataset
            suite2p_ds = flz.get_datasets(
                origin_name=sess_name,
                dataset_type="suite2p_rois",
                project_id=project,
                flexilims_session=flexilims_session,
                return_dataseries=False,
                filter_datasets={"anatomical_only": 3, 'annotated':True, 'ast_neuropil':False},
            )
            frame_rate = suite2p_ds[0].extra_attributes["fs"]

            # Sync to get trials_df with MotorSpeed
            _, trials_df = treadmill.sync_all_recordings(
                session_name=sess_name,
                flexilims_session=flexilims_session,
                project=project,
                filter_datasets={"anatomical_only": 3, 'annotated':True, 'ast_neuropil':False},
                recording_type="two_photon",
                photodiode_protocol=5,
                return_volumes=True,
                acceleration_time=None,
                cut_trial_end=None,
                trial_duration=None,
            )

            # Extract per-trial MotorSpeed
            # MotorSpeed_stim is an array of per-frame motor speeds during stim
            for _, trial in trials_df.iterrows():
                if "MotorSpeed_stim" in trial.index and trial["MotorSpeed_stim"] is not None:
                    ms = np.nanmedian(trial["MotorSpeed_stim"])
                elif "MotorSps_stim" in trial.index and trial["MotorSps_stim"] is not None:
                    ms = np.round(treadmill.sps2speed(np.nanmedian(trial["MotorSps_stim"])))
                else:
                    ms = np.nan

                motor_speed_records.append({
                    "session": sess_name,
                    "trial_no": trial["trial_no"],
                    "motor_speed": ms,
                    "frame_rate": frame_rate,
                })
            print(f"  {sess_name}: {len(trials_df)} trials, FR={frame_rate:.1f} Hz")
        except Exception as e:
            print(f"  ERROR syncing {sess_name}: {e}")

motor_speed_df = pd.DataFrame(motor_speed_records)

# Merge motor speed into predictions
predictions_df = predictions_df.merge(motor_speed_df, on=["session", "trial_no"], how="left")

print(f"\nMotor speed distribution:")
print(predictions_df["motor_speed"].value_counts().sort_index())

In [ ]:
trials_df.columns

In [ ]:
sess_df = predictions_df[predictions_df.session ==predictions_df.session.iloc[0] ]
by_gp = sess_df.groupby('motor_speed', 'target_of')

In [ ]:
# Plot prediction vs time and actual vs time for RS for 1 example session and 1 motor speed

example_session = "PZAG22.1a_S20260206"
example_speed = 15.0

# Filter predictions for the example session and motor speed
mask = (predictions_df["session"] == example_session) & (predictions_df["motor_speed"] == example_speed)
example_df = predictions_df[mask]

# Get frame rate for the session
fr = example_df["frame_rate"].iloc[0]

# Retrieve prediction and true arrays
preds = example_df["ridge_pred_RS_stim_closedloop"].tolist()
trues = example_df["ridge_true_RS_stim_closedloop"].tolist()

# Convert arrays to time series
time_bins = []
pred_vals = []
true_vals = []

for p, t in zip(preds, trues):
    p = np.asarray(p).ravel()
    t = np.asarray(t).ravel()
    if len(p) < 2 or len(t) < 2:
        continue
    time_bins.append(np.arange(len(p)) / fr)
    pred_vals.append(p)
    true_vals.append(t)

# Find the maximum length/time to average
max_len = max(len(x) for x in pred_vals)
common_time = np.arange(max_len) / fr

pred_interp = np.full((len(pred_vals), max_len), np.nan)
true_interp = np.full((len(true_vals), max_len), np.nan)

for i, (t_bin, p, t) in enumerate(zip(time_bins, pred_vals, true_vals)):
    pred_interp[i, :len(p)] = p
    true_interp[i, :len(t)] = t

# Calculate mean and SEM
mean_pred = np.nanmean(pred_interp, axis=0)
sem_pred = np.nanstd(pred_interp, axis=0) / np.sqrt(np.sum(~np.isnan(pred_interp), axis=0))
mean_true = np.nanmean(true_interp, axis=0)
sem_true = np.nanstd(true_interp, axis=0) / np.sqrt(np.sum(~np.isnan(true_interp), axis=0))

# Plot
plt.figure(figsize=(10, 6))

# Plot individual trials in light colors
for i in range(len(pred_vals)):
    plt.plot(time_bins[i], true_vals[i], color="gray", alpha=0.15, linewidth=0.8)
    plt.plot(time_bins[i], pred_vals[i], color="C0", alpha=0.15, linewidth=0.8)

# Plot mean +/- SEM
plt.plot(common_time, mean_true, color="black", linewidth=2.5, label="Actual RS")
plt.fill_between(common_time, mean_true - sem_true, mean_true + sem_true, color="black", alpha=0.15)

plt.plot(common_time, mean_pred, color="C0", linewidth=2.5, label="Predicted RS")
plt.fill_between(common_time, mean_pred - sem_pred, mean_pred + sem_pred, color="C0", alpha=0.15)

# Labeling
plt.title(f"Example Session: {example_session} | Motor Speed: {example_speed} cm/s (N={len(pred_vals)} trials)", fontsize=14, fontweight="bold")
plt.xlabel("Time from trial start (s)", fontsize=12)
plt.ylabel("Running Speed (cm/s)", fontsize=12)
plt.axhline(0, color="gray", linewidth=0.5, linestyle="--")

# Draw vertical dashed line for ramp-up period end
acceleration_time = 0.13
ramp_end = acceleration_time * example_speed
plt.axvline(ramp_end, color="red", linestyle=":", alpha=0.7, label=f"Ramp end (~{ramp_end:.2f}s)")
plt.legend(fontsize=11, loc="upper right")

plt.tight_layout()
plt.savefig("ridge_rs_example_prediction.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure to ridge_rs_example_prediction.png")

In [ ]:
# Cell 4: Define compute_timepoint_r2 functions

def compute_timepoint_r2(pred_arrays, true_arrays, n_bins=100):
    """Compute R² at each normalized time bin across a group of trials.

    For each trial:
        1. Interpolate pred and true arrays to n_bins points on [0, 1]

    At each time bin t, across all trials:
        R²(t) = 1 - Σ(pred(t) - true(t))² / Σ(true(t) - mean(true(t)))²

    Args:
        pred_arrays: list of 1-D arrays (one per trial)
        true_arrays: list of 1-D arrays (one per trial)
        n_bins: number of bins on [0, 1]

    Returns:
        time_frac (array): normalized time grid [0, 1]
        r2 (array): R² at each time bin
        n_valid (array): number of valid trials per bin
    """
    time_frac = np.linspace(0, 1, n_bins)
    # Interpolate all trials onto the common grid
    pred_interp = np.full((len(pred_arrays), n_bins), np.nan)
    true_interp = np.full((len(true_arrays), n_bins), np.nan)

    for i, (p, t) in enumerate(zip(pred_arrays, true_arrays)):
        p = np.asarray(p).ravel()
        t = np.asarray(t).ravel()
        if len(p) < 2 or len(t) < 2:
            continue
        trial_time = np.linspace(0, 1, len(p))
        pred_interp[i] = interp1d(trial_time, p, kind="linear", fill_value="extrapolate")(time_frac)
        true_interp[i] = interp1d(trial_time, t, kind="linear", fill_value="extrapolate")(time_frac)

    # Compute R² at each time bin across trials
    r2 = np.full(n_bins, np.nan)
    n_valid = np.zeros(n_bins, dtype=int)
    for j in range(n_bins):
        valid = ~(np.isnan(pred_interp[:, j]) | np.isnan(true_interp[:, j]))
        n_valid[j] = valid.sum()
        if n_valid[j] < 3:
            continue
        p_vals = pred_interp[valid, j]
        t_vals = true_interp[valid, j]
        ss_res = np.sum((t_vals - p_vals) ** 2)
        ss_tot = np.sum((t_vals - t_vals.mean()) ** 2)
        if ss_tot > 0:
            r2[j] = 1 - ss_res / ss_tot

    return time_frac, r2, n_valid


def compute_timepoint_r2_absolute(pred_arrays, true_arrays, frame_rates, dt=0.5, max_time=30):
    """Compute R² at each absolute time bin (in seconds) across trials.

    Same logic as compute_timepoint_r2 but uses absolute time.

    Args:
        pred_arrays: list of 1-D arrays (one per trial)
        true_arrays: list of 1-D arrays (one per trial)
        frame_rates: list/array of frame rates (one per trial)
        dt: time bin width in seconds
        max_time: maximum time in seconds

    Returns:
        time_sec (array): absolute time grid in seconds
        r2 (array): R² at each time bin
        n_valid (array): number of valid trials per bin
    """
    time_sec = np.arange(0, max_time, dt)
    n_bins = len(time_sec)

    pred_interp = np.full((len(pred_arrays), n_bins), np.nan)
    true_interp = np.full((len(true_arrays), n_bins), np.nan)

    for i, (p, t, fr) in enumerate(zip(pred_arrays, true_arrays, frame_rates)):
        p = np.asarray(p).ravel()
        t = np.asarray(t).ravel()
        if len(p) < 2 or len(t) < 2:
            continue
        trial_time = np.arange(len(p)) / fr
        max_trial_time = trial_time[-1]
        valid_bins = time_sec <= max_trial_time
        if valid_bins.sum() < 2:
            continue
        pred_interp[i, valid_bins] = interp1d(trial_time, p, kind="linear", fill_value=np.nan, bounds_error=False)(time_sec[valid_bins])
        true_interp[i, valid_bins] = interp1d(trial_time, t, kind="linear", fill_value=np.nan, bounds_error=False)(time_sec[valid_bins])

    r2 = np.full(n_bins, np.nan)
    n_valid = np.zeros(n_bins, dtype=int)
    for j in range(n_bins):
        valid = ~(np.isnan(pred_interp[:, j]) | np.isnan(true_interp[:, j]))
        n_valid[j] = valid.sum()
        if n_valid[j] < 3:
            continue
        p_vals = pred_interp[valid, j]
        t_vals = true_interp[valid, j]
        ss_res = np.sum((t_vals - p_vals) ** 2)
        ss_tot = np.sum((t_vals - t_vals.mean()) ** 2)
        if ss_tot > 0:
            r2[j] = 1 - ss_res / ss_tot

    return time_sec, r2, n_valid

In [ ]:
# Cell 5: Compute R² across time for all targets

TARGETS = ["RS_stim", "OF_stim", "depth"]
CONDITION = "closedloop"  # motor sessions are closedloop by definition
ACCELERATION_TIME = 0.13  # s per cm/s (from treadmill.py default)

# Get all non-zero motor speeds
all_speeds = sorted(predictions_df.loc[
    predictions_df["motor_speed"] > 0, "motor_speed"
].dropna().unique())
print(f"Motor speeds (excluding 0): {all_speeds}")

# Store results
results_normalized = {}  # {(target, speed): (time_frac, r2, n_valid)}
results_absolute = {}    # {(target, speed): (time_sec, r2, n_valid)}

for target in TARGETS:
    pred_col = f"ridge_pred_{target}_{CONDITION}"
    true_col = f"ridge_true_{target}_{CONDITION}"

    if pred_col not in predictions_df.columns:
        print(f"  Skipping {target}: column {pred_col} not found")
        continue

    for speed in all_speeds:
        mask = (
            (predictions_df["motor_speed"] == speed)
            & predictions_df[pred_col].notna()
            & predictions_df[true_col].notna()
        )
        sub = predictions_df[mask]
        if len(sub) < 5:
            print(f"  {target} @ {speed} cm/s: only {len(sub)} trials, skipping")
            continue

        pred_arrays = sub[pred_col].tolist()
        true_arrays = sub[true_col].tolist()
        frame_rates = sub["frame_rate"].values

        # Normalized time
        tf, r2_norm, nv = compute_timepoint_r2(pred_arrays, true_arrays)
        results_normalized[(target, speed)] = (tf, r2_norm, nv)

        # Absolute time
        ts, r2_abs, nv_abs = compute_timepoint_r2_absolute(
            pred_arrays, true_arrays, frame_rates
        )
        results_absolute[(target, speed)] = (ts, r2_abs, nv_abs)

        print(f"  {target} @ {speed:5.1f} cm/s: {len(sub)} trials")

print(f"\nTotal result entries: {len(results_normalized)} normalized, {len(results_absolute)} absolute")

In [ ]:
# Cell 6: Plot 3×2 grid  (3 targets × 2 time axes)
#
#            Normalized time [0,1]    |    Absolute time (seconds)
# RS_stim    [R² vs frac, by speed]   |    [R² vs seconds, by speed]
# OF_stim    [R² vs frac, by speed]   |    [R² vs seconds, by speed]
# depth      [R² vs frac, by speed]   |    [R² vs seconds, by speed]

TARGET_LABELS = {
    "RS_stim": "Running Speed (RS)",
    "OF_stim": "Optic Flow (OF)",
    "depth": "Depth",
}

# Colormap for motor speeds (sequential)
speed_norm = plt.Normalize(vmin=min(all_speeds), vmax=max(all_speeds))
speed_cmap = plt.cm.plasma

fig, axes = plt.subplots(3, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle("Ridge Decoder R² Across Trial Time by Motor Speed", fontsize=16, fontweight="bold")

for row, target in enumerate(TARGETS):
    ax_norm = axes[row, 0]
    ax_abs = axes[row, 1]

    # --- Left column: normalized time ---
    for speed in all_speeds:
        key = (target, speed)
        if key not in results_normalized:
            continue
        tf, r2, nv = results_normalized[key]
        color = speed_cmap(speed_norm(speed))
        ax_norm.plot(tf, r2, color=color, linewidth=2, label=f"{speed:.0f} cm/s")

    ax_norm.set_ylabel(f"{TARGET_LABELS[target]}\nR²")
    ax_norm.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    if row == 2:
        ax_norm.set_xlabel("Normalized trial time")
    if row == 0:
        ax_norm.set_title("Normalized time [0, 1]")
        ax_norm.legend(fontsize=8, loc="upper left", ncol=2, framealpha=0.8)

    # --- Right column: absolute time ---
    for speed in all_speeds:
        key = (target, speed)
        if key not in results_absolute:
            continue
        ts, r2, nv = results_absolute[key]
        color = speed_cmap(speed_norm(speed))
        # Only plot where we have enough trials
        valid = nv >= 3
        ax_abs.plot(ts[valid], r2[valid], color=color, linewidth=2, label=f"{speed:.0f} cm/s")

        # Vertical dashed line at approximate ramp-up end time
        ramp_end = ACCELERATION_TIME * speed
        ax_abs.axvline(ramp_end, color=color, linewidth=1, linestyle=":", alpha=0.5)

    ax_abs.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    if row == 2:
        ax_abs.set_xlabel("Time from trial start (s)")
    if row == 0:
        ax_abs.set_title("Absolute time (seconds)")
        ax_abs.legend(fontsize=8, loc="upper left", ncol=2, framealpha=0.8)

plt.savefig("ridge_r2_across_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure to ridge_r2_across_time.png")